In [ ]:
import os
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE') # Crash fix bc le torch
os.environ.setdefault('OMP_NUM_THREADS', '1')

import numpy as np

# xgboost 1.7.x still references np.NaN internally (removed in numpy>=2.0, fixed only in
# xgboost>=2.1) -- QuantileDMatrix construction on categorical columns crashes with
# "AttributeError: `np.NaN` was removed in the NumPy 2.0 release" without this shim.
if not hasattr(np, 'NaN'):
    np.NaN = np.nan

import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.metrics import (classification_report, roc_auc_score, average_precision_score,
                              roc_curve, auc, precision_recall_curve)
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_sample_weight, compute_class_weight

import xgboost as xgb
import lightgbm as gbm
import shap

import torch
torch.set_num_threads(1)  # keep torch off the OpenMP pool xgboost/lightgbm are also using
import torch.nn as nn
import torch.nn.functional as F
import skorch
from functools import partial
from skorch.callbacks import Callback, EarlyStopping, GradientNormClipping

os.makedirs('../models_H3', exist_ok=True)
MODELS_DIR = '../models_H3'

In [ ]:
IBAN_COL   = 'Account'
TS_COL     = 'Timestamp'
LABEL_COL  = 'proxy_label'
DECLARING_PSP_COL  = 'From Bank'
HOLDING_PSP_COL    = 'To Bank'
REPORTING_BANK_COL = 'bank'   # identifiant de la banque declarante dans le registre Fake-RF
N = 1

train_df = pd.read_parquet('train_df.parquet')
test_df  = pd.read_parquet('test_df.parquet')
print(f"train_df: {train_df.shape}, test_df: {test_df.shape}")

In [ ]:
COUNTERPARTY_COL = 'Account.1'
frf = pd.read_csv(
    '../dataset/fake_fncrf.csv',
    usecols=[IBAN_COL, COUNTERPARTY_COL, TS_COL, DECLARING_PSP_COL, HOLDING_PSP_COL, REPORTING_BANK_COL, LABEL_COL],
    on_bad_lines='warn', engine='python',
)
frf[TS_COL] = pd.to_datetime(frf[TS_COL], errors='coerce')
frf = frf.sort_values([IBAN_COL, TS_COL]).reset_index(drop=True)
print(f"fake_fncrf.csv: {frf.shape}, {frf[REPORTING_BANK_COL].nunique()} banques declarantes distinctes")
print(f"bank == From Bank pour {(frf[REPORTING_BANK_COL].astype(str) == frf[DECLARING_PSP_COL].astype(str)).mean():.6f} des lignes")

In [ ]:
# nb_banks_signalants / delai_inter_signalement_h : signal cross-PSP au niveau compte
first_report = (frf.groupby([IBAN_COL, REPORTING_BANK_COL])[TS_COL]
                    .first()
                    .reset_index())

signal_stats = (first_report.groupby(IBAN_COL)
                .agg(nb_banks_signalants=(REPORTING_BANK_COL, 'nunique'),
                     first_signal_ts=(TS_COL, 'min'),
                     last_signal_ts=(TS_COL, 'max'))
                .reset_index())
signal_stats['delai_inter_signalement_h'] = (
    (signal_stats['last_signal_ts'] - signal_stats['first_signal_ts']).dt.total_seconds() / 3600.0
)

print(signal_stats['nb_banks_signalants'].value_counts().sort_index())
n_multi = (signal_stats['nb_banks_signalants'] > 1).sum()
n_total = len(signal_stats)
print(f"{n_multi} comptes signales par plus d'une banque sur {n_total} ({100*n_multi/n_total:.4f} %)")

In [ ]:
# counterparty_flagged_by_other_bank : le compte contrepartie (Account.1) d'au moins une transaction de ce compte a-t-il ete declare 
# 'Fraudeur' par une banque DIFFERENTE de la bq ayant declare ce compte ?
fraud_by_account = frf.loc[frf[LABEL_COL] == 'Fraudeur'].groupby(IBAN_COL)[REPORTING_BANK_COL].apply(set)

def _flagged_by_other_bank(row):
    banks = fraud_by_account.get(row[COUNTERPARTY_COL])
    if not banks:
        return False
    return len(banks - {row[REPORTING_BANK_COL]}) > 0

frf['counterparty_flagged_other_bank_txn'] = frf.apply(_flagged_by_other_bank, axis=1)

counterparty_stats = (frf.groupby(IBAN_COL)
                      .agg(counterparty_flagged_by_other_bank=('counterparty_flagged_other_bank_txn', 'any'),
                           counterparty_other_bank_flag_rate=('counterparty_flagged_other_bank_txn', 'mean'))
                      .reset_index())
counterparty_stats['counterparty_flagged_by_other_bank'] = (
    counterparty_stats['counterparty_flagged_by_other_bank'].astype(int)
)

print(counterparty_stats['counterparty_flagged_by_other_bank'].value_counts())
n_flagged = counterparty_stats['counterparty_flagged_by_other_bank'].sum()
n_total_acc = len(counterparty_stats)
print(f"{n_flagged} comptes ({100*n_flagged/n_total_acc:.2f} %) ont au moins une contrepartie "
      f"deja declaree Fraudeur par une autre banque")

# fraud rates
acc_is_fraud = frf.groupby(IBAN_COL)[LABEL_COL].apply(lambda s: (s == 'Fraudeur').any()).rename('is_fraud').astype(int)
check = counterparty_stats.set_index(IBAN_COL).join(acc_is_fraud)
print(check.groupby('counterparty_flagged_by_other_bank')['is_fraud'].agg(['mean', 'count']))

In [ ]:
merge_cols = ['nb_banks_signalants', 'delai_inter_signalement_h']
counterparty_merge_cols = ['counterparty_flagged_by_other_bank', 'counterparty_other_bank_flag_rate']
for df in (train_df, test_df):
    df.drop(columns=[c for c in merge_cols + counterparty_merge_cols if c in df.columns], inplace=True)

train_df = train_df.merge(signal_stats[[IBAN_COL] + merge_cols], on=IBAN_COL, how='left')
test_df  = test_df.merge(signal_stats[[IBAN_COL] + merge_cols], on=IBAN_COL, how='left')

train_df = train_df.merge(counterparty_stats[[IBAN_COL] + counterparty_merge_cols], on=IBAN_COL, how='left')
test_df  = test_df.merge(counterparty_stats[[IBAN_COL] + counterparty_merge_cols], on=IBAN_COL, how='left')

for df in (train_df, test_df):
    df['nb_banks_signalants'] = df['nb_banks_signalants'].fillna(1)
    df['delai_inter_signalement_h'] = df['delai_inter_signalement_h'].fillna(0.0)
    df['counterparty_flagged_by_other_bank'] = df['counterparty_flagged_by_other_bank'].fillna(0)
    df['counterparty_other_bank_flag_rate'] = df['counterparty_other_bank_flag_rate'].fillna(0.0)

In [ ]:
# Variables cross-institutionnelles 
CROSS_INST_COLS = [
    'nb_banks_signalants', 'delai_inter_signalement_h',
    'counterparty_flagged_by_other_bank', 'counterparty_other_bank_flag_rate',
    'declaring.fraud_rate_lag1', 'declaring.fp_rate_lag1',
    'declaring.nb.prior.txn_lag1', 'declaring.has.history_lag1',
    'holding.fraud_rate_lag1', 'holding.fp_rate_lag1',
    'holding.nb.prior.txn_lag1', 'holding.has.history_lag1',
    'corridor.fraud_rate_lag1', 'corridor.nb.prior.txn_lag1',
    'nb.distinct.to.bank_cum_lag1', 'nb.distinct.from.bank_cum_lag1',
    'nb.iban.holder_lag1', 'nb.events.holder_lag1',
    'nb.iban.declaring_lag1', 'nb.events.declaring_lag1',
    'top.1.holder.RC_lag1', 'top.1.holder.SC_lag1',
    'top.1.declaring.RC_lag1', 'top.1.declaring.SC_lag1',
    'From Bank_lag1', 'To Bank_lag1',
]
CROSS_INST_COLS = [c for c in CROSS_INST_COLS if c in train_df.columns]

DROP_ALWAYS = [LABEL_COL, IBAN_COL, TS_COL, 'key_lag1', 'Unnamed: 0_lag1', 'fold_lag1',
               'Amount Received_lag1', 'Amount Paid_lag1'] # comme H1 et H2

feature_cols_full    = [c for c in train_df.columns if c not in DROP_ALWAYS]
feature_cols_reduced = [c for c in feature_cols_full if c not in CROSS_INST_COLS]

print(f"{len(feature_cols_full)} variables (modele complet), "
      f"{len(feature_cols_reduced)} variables (modele reduit), "
      f"{len(CROSS_INST_COLS)} variables cross-institutionnelles retirees")

In [ ]:
y_train = train_df[LABEL_COL].astype(np.int64).values
y_test  = test_df[LABEL_COL].astype(np.int64).values

X_train_full    = train_df[feature_cols_full]
X_test_full     = test_df[feature_cols_full]
X_train_reduced = train_df[feature_cols_reduced]
X_test_reduced  = test_df[feature_cols_reduced]

cat_cols_full    = [c for c in feature_cols_full if train_df[c].dtype.name == 'category']
cat_cols_reduced = [c for c in feature_cols_reduced if train_df[c].dtype.name == 'category']
print(f"colonnes categorielles - complet: {len(cat_cols_full)}, reduit: {len(cat_cols_reduced)}")

In [ ]:
# Hyperparametres repris tels quels de la recherche bayesienne deja menee dans H1
XGB_BEST_PARAMS = dict(
    colsample_bytree=1.0, learning_rate=0.2034370190697667, max_delta_step=3,
    max_depth=10, min_child_weight=5, n_estimators=2000,
    reg_alpha=1e-09, reg_lambda=1e-09, subsample=0.8371868846738777,
)
LGBM_BEST_PARAMS = dict(
    colsample_bytree=0.5450117686859245, learning_rate=0.21661693127982293,
    max_depth=12, min_child_samples=74, n_estimators=1000, num_leaves=1000,
    reg_alpha=1.1515833248696066, reg_lambda=0.20493132534038447,
    subsample=0.7598010881577231,
)

def evaluate(model, X_te, y_te, name):
    proba = model.predict_proba(X_te)[:, 1]
    preds = model.predict(X_te)
    report = classification_report(y_te, preds, output_dict=True)
    roc = roc_auc_score(y_te, proba)
    pr_auc = average_precision_score(y_te, proba)
    print(f"--- {name} ---")
    print(classification_report(y_te, preds))
    print(f"ROC-AUC: {roc:.4f} | PR-AUC: {pr_auc:.4f}")
    return {'name': name, 'roc_auc': roc, 'pr_auc': pr_auc,
            'precision_1': report['1']['precision'], 'recall_1': report['1']['recall'],
            'f1_1': report['1']['f1-score']}


results = []
models = {}

In [ ]:
import time
t0 = time.time()
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)
models['xgb_full'] = xgb.XGBClassifier(
    enable_categorical=True, tree_method='hist', eval_metric='aucpr', random_state=42,
    n_jobs=-1, **XGB_BEST_PARAMS,
)
models['xgb_full'].fit(X_train_full, y_train, sample_weight=sample_weights)
print(f"xgb_full fit in {time.time() - t0:.1f}s")
results.append(evaluate(models['xgb_full'], X_test_full, y_test, 'xgb_full'))
joblib.dump(models['xgb_full'], f'{MODELS_DIR}/model_xgb_full_H3.pkl')

In [ ]:
t0 = time.time()
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)
models['xgb_reduced'] = xgb.XGBClassifier(
    enable_categorical=True, tree_method='hist', eval_metric='aucpr', random_state=42,
    n_jobs=-1, **XGB_BEST_PARAMS,
)
models['xgb_reduced'].fit(X_train_reduced, y_train, sample_weight=sample_weights)
print(f"xgb_reduced fit in {time.time() - t0:.1f}s")
results.append(evaluate(models['xgb_reduced'], X_test_reduced, y_test, 'xgb_reduced'))
joblib.dump(models['xgb_reduced'], f'{MODELS_DIR}/model_xgb_reduced_H3.pkl')

In [ ]:
t0 = time.time()
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)
models['lgbm_full'] = gbm.LGBMClassifier(objective='binary', random_state=42, n_jobs=-1, **LGBM_BEST_PARAMS)
models['lgbm_full'].fit(X_train_full, y_train, sample_weight=sample_weights)
print(f"lgbm_full fit in {time.time() - t0:.1f}s")
results.append(evaluate(models['lgbm_full'], X_test_full, y_test, 'lgbm_full'))
joblib.dump(models['lgbm_full'], f'{MODELS_DIR}/model_lightgbm_full_H3.pkl')

In [ ]:
t0 = time.time()
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)
models['lgbm_reduced'] = gbm.LGBMClassifier(objective='binary', random_state=42, n_jobs=-1, **LGBM_BEST_PARAMS)
models['lgbm_reduced'].fit(X_train_reduced, y_train, sample_weight=sample_weights)
print(f"lgbm_reduced fit in {time.time() - t0:.1f}s")
results.append(evaluate(models['lgbm_reduced'], X_test_reduced, y_test, 'lgbm_reduced'))
joblib.dump(models['lgbm_reduced'], f'{MODELS_DIR}/model_lightgbm_reduced_H3.pkl')

In [ ]:
MLP_BEST_PARAMS = dict(lr=9e-4, dim=128, emb_dim=8, mlp_mult=2, depth=1, max_epochs=50)


class CastedLinear(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(out_dim, in_dim))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, x):
        return F.linear(x, self.weight.to(x.dtype))


class MLPBlock(nn.Module):
    def __init__(self, dim, mlp_mult):
        super().__init__()
        hidden = dim * mlp_mult
        self.gate = CastedLinear(dim, hidden)
        self.up   = CastedLinear(dim, hidden)
        self.proj = CastedLinear(hidden, dim)

    def forward(self, x):
        return self.proj(F.relu(self.gate(x)) * self.up(x))


class MLPClassifier(nn.Module):
    def __init__(self, num_cont, cat_cardinalities, dim=64, mlp_mult=4, depth=2, emb_dim=8):
        super().__init__()
        self.num_cont = num_cont
        self.num_cat = len(cat_cardinalities)
        self.embeddings = nn.ModuleList([nn.Embedding(card, emb_dim) for card in cat_cardinalities])
        in_dim = num_cont + emb_dim * len(cat_cardinalities)
        self.input_norm = nn.LayerNorm(in_dim)
        self.input_proj = CastedLinear(in_dim, dim)
        self.blocks = nn.ModuleList([MLPBlock(dim, mlp_mult) for _ in range(depth)])
        self.head = nn.Linear(dim, 2)
        self.norms = nn.ModuleList([nn.LayerNorm(dim) for _ in range(depth)])

    def forward(self, x):
        x_cont = x[:, :self.num_cont].float()
        x_cat  = x[:, self.num_cont:].long().clamp(min=0)
        embs = [e(x_cat[:, i].clamp(0, e.num_embeddings - 1)) for i, e in enumerate(self.embeddings)]
        x = torch.cat([x_cont] + embs, dim=-1)
        x = self.input_norm(x)
        x = self.input_proj(x)
        for block, norm in zip(self.blocks, self.norms):
            x = x + block(norm(x))
        return self.head(x).squeeze(-1)


class DynamicClassWeights(Callback):
    def on_train_begin(self, net, X=None, y=None, **kwargs):
        classes = np.unique(y)
        weights = compute_class_weight('balanced', classes=classes, y=y)
        net.criterion_.weight = torch.tensor(weights, dtype=torch.float32)


def build_mlp_arrays(X_tr, X_te, cat_cols, num_cols):
    # Continuous block: median-impute + standardize, fit on train only.
    num_imputer = SimpleImputer(strategy='median')
    num_scaler = StandardScaler()
    if num_cols:
        num_train = num_scaler.fit_transform(num_imputer.fit_transform(X_tr[num_cols]))
        num_test  = num_scaler.transform(num_imputer.transform(X_te[num_cols]))
    else:
        num_train = np.empty((len(X_tr), 0))
        num_test  = np.empty((len(X_te), 0))

    # Categorical block: integer codes from the TRAIN categories (same convention as H1's
    # embeddings, which clamp negative/unseen codes to 0 in MLPClassifier.forward).
    cat_cardinalities = []
    cat_train_cols, cat_test_cols = [], []
    for c in cat_cols:
        categories = X_tr[c].cat.categories
        cat_cardinalities.append(max(len(categories), 1))
        cat_train_cols.append(X_tr[c].cat.codes.to_numpy(dtype=np.float32))
        cat_test_cols.append(pd.Categorical(X_te[c], categories=categories).codes.astype(np.float32))
    cat_train = np.stack(cat_train_cols, axis=1) if cat_cols else np.empty((len(X_tr), 0))
    cat_test  = np.stack(cat_test_cols, axis=1) if cat_cols else np.empty((len(X_te), 0))

    X_train_np = np.hstack([num_train, cat_train]).astype(np.float32)
    X_test_np  = np.hstack([num_test, cat_test]).astype(np.float32)
    return X_train_np, X_test_np, cat_cardinalities


def fit_mlp(X_tr, X_te, y_tr, cat_cols, num_cols):
    X_train_np, X_test_np, cat_cardinalities = build_mlp_arrays(X_tr, X_te, cat_cols, num_cols)
    FixedMLPClassifier = partial(
        MLPClassifier, num_cont=len(num_cols), cat_cardinalities=cat_cardinalities,
        dim=MLP_BEST_PARAMS['dim'], mlp_mult=MLP_BEST_PARAMS['mlp_mult'],
        depth=MLP_BEST_PARAMS['depth'], emb_dim=MLP_BEST_PARAMS['emb_dim'],
    )
    net = skorch.NeuralNetClassifier(
        module=FixedMLPClassifier,
        max_epochs=MLP_BEST_PARAMS['max_epochs'],
        lr=MLP_BEST_PARAMS['lr'],
        batch_size=1024,
        iterator_train__shuffle=True,
        train_split=skorch.dataset.ValidSplit(0.2, stratified=True),
        verbose=1,
        callbacks=[
            DynamicClassWeights(),
            GradientNormClipping(gradient_clip_value=1.0),
            EarlyStopping(monitor='valid_loss', patience=5, lower_is_better=True),
        ],
        criterion=nn.CrossEntropyLoss,
    )
    net.fit(X_train_np, y_tr.astype(np.int64))
    return net, X_test_np


mlp_test_np = {}

In [ ]:
t0 = time.time()
torch.manual_seed(0)
models['mlp_full'], mlp_test_np['full'] = fit_mlp(
    X_train_full, X_test_full, y_train, cat_cols_full,
    [c for c in feature_cols_full if c not in cat_cols_full],
)
print(f"mlp_full fit in {time.time() - t0:.1f}s")
results.append(evaluate(models['mlp_full'], mlp_test_np['full'], y_test, 'mlp_full'))
joblib.dump(models['mlp_full'], f'{MODELS_DIR}/model_mlp_full_H3.pkl')
joblib.dump(mlp_test_np['full'], f'{MODELS_DIR}/mlp_test_np_full_H3.pkl')

In [ ]:
t0 = time.time()
torch.manual_seed(0)
models['mlp_reduced'], mlp_test_np['reduced'] = fit_mlp(
    X_train_reduced, X_test_reduced, y_train, cat_cols_reduced,
    [c for c in feature_cols_reduced if c not in cat_cols_reduced],
)
print(f"mlp_reduced fit in {time.time() - t0:.1f}s")
results.append(evaluate(models['mlp_reduced'], mlp_test_np['reduced'], y_test, 'mlp_reduced'))
joblib.dump(models['mlp_reduced'], f'{MODELS_DIR}/model_mlp_reduced_H3.pkl')
joblib.dump(mlp_test_np['reduced'], f'{MODELS_DIR}/mlp_test_np_reduced_H3.pkl')

In [ ]:
target_precision = 0.75
for name, model, X_te in [('mlp_full', models['mlp_full'], mlp_test_np['full']),
                           ('mlp_reduced', models['mlp_reduced'], mlp_test_np['reduced'])]:
    probs = model.predict_proba(X_te)[:, 1]
    precision, recall, thresholds = precision_recall_curve(y_test, probs)
    idx = np.argmax(precision >= target_precision)
    best_thresh = thresholds[idx]
    print(f"--- {name} (seuil pour precision >= {target_precision}) ---")
    print(f"threshold={best_thresh:.4f}  precision={precision[idx]:.4f}  recall={recall[idx]:.4f}")

    y_pred = (probs >= best_thresh).astype(int)
    print(classification_report(y_test, y_pred))
    print(f"ROC-AUC: {roc_auc_score(y_test, probs):.4f}")

In [ ]:
# Sanity check: reload the saved XGBoost (full) model from disk and confirm it evaluates
# identically to the in-memory model above.
loaded_xgb_full = joblib.load(f'{MODELS_DIR}/model_xgb_full_H3.pkl')
evaluate(loaded_xgb_full, X_test_full, y_test, 'xgb_full (reloaded from disk)')

In [ ]:
results_df = pd.DataFrame(results).set_index('name')
results_df['delta_roc_auc_vs_reduced'] = np.nan
for algo in ('xgb', 'lgbm', 'mlp'):
    full_roc = results_df.loc[f'{algo}_full', 'roc_auc']
    red_roc  = results_df.loc[f'{algo}_reduced', 'roc_auc']
    results_df.loc[f'{algo}_full', 'delta_roc_auc_vs_reduced'] = full_roc - red_roc
results_df

In [ ]:
models = {
    'xgb_full':     joblib.load(f'{MODELS_DIR}/model_xgb_full_H3.pkl'),
    'xgb_reduced':  joblib.load(f'{MODELS_DIR}/model_xgb_reduced_H3.pkl'),
    'lgbm_full':    joblib.load(f'{MODELS_DIR}/model_lightgbm_full_H3.pkl'),
    'lgbm_reduced': joblib.load(f'{MODELS_DIR}/model_lightgbm_reduced_H3.pkl'),
    'mlp_full':     joblib.load(f'{MODELS_DIR}/model_mlp_full_H3.pkl'),
    'mlp_reduced':  joblib.load(f'{MODELS_DIR}/model_mlp_reduced_H3.pkl'),
}
mlp_test_np = {
    'full':    joblib.load(f'{MODELS_DIR}/mlp_test_np_full_H3.pkl'),
    'reduced': joblib.load(f'{MODELS_DIR}/mlp_test_np_reduced_H3.pkl'),
}
print('modeles recharges depuis', MODELS_DIR, '-', list(models.keys()))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for name, model, X_te in [
    ('XGBoost (complet)', models['xgb_full'], X_test_full),
    ('XGBoost (reduit)', models['xgb_reduced'], X_test_reduced),
    ('LightGBM (complet)', models['lgbm_full'], X_test_full),
    ('LightGBM (reduit)', models['lgbm_reduced'], X_test_reduced),
    ('MLP (complet)', models['mlp_full'], mlp_test_np['full']),
    ('MLP (reduit)', models['mlp_reduced'], mlp_test_np['reduced']),
]:
    proba = model.predict_proba(X_te)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, proba)
    ax.plot(fpr, tpr, lw=2, label=f'{name} (AUC = {auc(fpr, tpr):.4f})')
ax.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--', label='Hasard')
ax.set_xlabel('Taux de faux positifs')
ax.set_ylabel('Taux de vrais positifs')
ax.set_title('H3 - ROC, avec vs. sans variables cross-institutionnelles')
ax.legend(loc='lower right')
ax.grid(alpha=0.3)
plt.show()

In [ ]:
# SHAP sur les modeles complets
def shap_importance_table(model, X_bg, feature_cols):
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_bg)
    if isinstance(shap_values, list):
        shap_values = shap_values[1]
    mean_abs = np.abs(shap_values).mean(axis=0)
    imp = pd.DataFrame({'feature': feature_cols, 'mean_abs_shap': mean_abs})
    imp['is_cross_institutional'] = imp['feature'].isin(CROSS_INST_COLS)
    return imp.sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

shap_xgb  = shap_importance_table(models['xgb_full'],  X_test_full, feature_cols_full)
shap_lgbm = shap_importance_table(models['lgbm_full'], X_test_full, feature_cols_full)

print("=== XGBoost - top 15 variables (SHAP) ===")
print(shap_xgb.head(15))
print()
print("=== LightGBM - top 15 variables (SHAP) ===")
print(shap_lgbm.head(15))

In [ ]:
for name, imp in [('XGBoost', shap_xgb), ('LightGBM', shap_lgbm)]:
    share = imp.groupby('is_cross_institutional')['mean_abs_shap'].sum()
    share = share / share.sum()
    print(f"{name} - part de |SHAP| moyen portee par les variables cross-institutionnelles: "
          f"{share.get(True, 0.0):.4f}")
    rank_cross = imp.index[imp['is_cross_institutional']].tolist()
    print(f"  rangs (0 = plus important) des variables cross-institutionnelles: {rank_cross}")

In [ ]:

OWN_BANK_KEYED = [
    'declaring.fraud_rate_lag1', 'declaring.fp_rate_lag1',
    'declaring.nb.prior.txn_lag1', 'declaring.has.history_lag1',
    'top.1.declaring.RC_lag1', 'top.1.declaring.SC_lag1',
    'nb.iban.declaring_lag1', 'nb.events.declaring_lag1',
    'corridor.fraud_rate_lag1', 'corridor.nb.prior.txn_lag1',
]
COUNTERPARTY_KEYED = [
    'holding.fraud_rate_lag1', 'holding.fp_rate_lag1',
    'holding.nb.prior.txn_lag1', 'holding.has.history_lag1',
    'top.1.holder.RC_lag1', 'top.1.holder.SC_lag1',
    'nb.iban.holder_lag1', 'nb.events.holder_lag1',
]
ACCOUNT_CROSS_BANK_KEYED = [
    'nb.distinct.from.bank_cum_lag1', 'nb_banks_signalants', 'delai_inter_signalement_h',
    'counterparty_flagged_by_other_bank', 'counterparty_other_bank_flag_rate',
]

In [ ]:
# on recalc par bq
declaring_key = frf[DECLARING_PSP_COL].fillna('UNKNW')
corridor_key  = declaring_key.astype(str) + '_' + frf[HOLDING_PSP_COL].fillna('UNKNW').astype(str)

global_declaring_prior_n = declaring_key.groupby(declaring_key).cumcount()
local_declaring_prior_n = pd.concat([
    g[DECLARING_PSP_COL].fillna('UNKNW').pipe(lambda k: k.groupby(k).cumcount())
    for _, g in frf.groupby(REPORTING_BANK_COL)
]).sort_index()

global_corridor_prior_n = corridor_key.groupby(corridor_key).cumcount()
local_corridor_prior_n = pd.concat([
    (g[DECLARING_PSP_COL].fillna('UNKNW').astype(str) + '_' + g[HOLDING_PSP_COL].fillna('UNKNW').astype(str))
    .pipe(lambda k: k.groupby(k).cumcount())
    for _, g in frf.groupby(REPORTING_BANK_COL)
]).sort_index()

print("declaring.nb.prior.txn : local == global pour toutes les lignes :",
      (global_declaring_prior_n.values == local_declaring_prior_n.values).all())
print("corridor.nb.prior.txn  : local == global pour toutes les lignes :",
      (global_corridor_prior_n.values == local_corridor_prior_n.values).all())

In [ ]:
# la meme
holding_key = frf[HOLDING_PSP_COL].fillna('UNKNW')
global_holding_prior_n = holding_key.groupby(holding_key).cumcount()

visible_mask = global_holding_prior_n > 0
share_visible_locally = global_corridor_prior_n[visible_mask] / global_holding_prior_n[visible_mask]
print("holding.nb.prior.txn : part de l'historique visible localement via le corridor propre au client")
print(share_visible_locally.describe())
print(f"part des lignes ou l'historique 'holding' n'est que partiellement visible localement: "
      f"{(share_visible_locally < 0.999).mean():.4f}")

In [ ]:
# 4) counterparty_flagged_by_other_bank
local_counterparty_flag = 0  # constant (False) par construction pour tout client federe
mismatch_counterparty = (counterparty_stats['counterparty_flagged_by_other_bank'] != local_counterparty_flag).sum()
print(f"comptes ou counterparty_flagged_by_other_bank local (=0, par construction) diverge du "
      f"centralise: {mismatch_counterparty} / {len(counterparty_stats)} "
      f"({100*mismatch_counterparty/len(counterparty_stats):.2f} %)")
print(check.groupby('counterparty_flagged_by_other_bank')['is_fraud'].agg(['mean', 'count']))